# TeachAgent 4-Turn Demo: High School Sequence Problem

这个 notebook 演示一个高中数列题的 4 轮教学过程，重点观察：

- 每轮 `quality` / `understands`
- `mode` 是否发生变化
- `turn_index` / `max_turns`
- `stop_reason` 何时触发


In [ ]:
%pip install -U azure-ai-projects azure-identity openai nest_asyncio pydantic

In [ ]:
import sys
import importlib

sys.path.append("/Users/xumuchi/Desktop/TeachAgent")

import coach_agent
importlib.reload(coach_agent)

print(coach_agent.diagnose_environment())
print("coach_agent_version:", coach_agent.COACH_AGENT_VERSION)

In [ ]:
agent = coach_agent.FoundryCoachAgent()

session = agent.create_session(
    problem_text=(
        "已知等差数列 {a_n} 中，a_1=3，a_4=12。"
        "求公差 d，并求 a_10 的值。"
    ),
    error_type=coach_agent.ErrorType.MISSING_STRATEGY,
    student_profile="学生知道等差数列公式，但经常不知道先从哪个条件下手。",
    extra_rule="先引导学生找中间量，再决定是否总结，不要一开始直接给完整答案。",
    max_turns=4,
)

print("session.max_turns =", session.max_turns)

In [ ]:
ANALYSIS_SOURCE_LABELS = {
    "ai_tool_pydantic_parse": "官方 structured outputs + Pydantic parse",
    "ai_tool_json_mode": "json_object 模式成功",
    "ai_tool_text_json_validated": "普通文本 JSON，被 Pydantic 校验通过",
    "ai_tool_text_json_basic": "普通文本 JSON，只通过基础 JSON 解析",
    "fallback_heuristic": "本地 heuristic fallback",
}

def run_turn(student_reply, stream=False):
    if stream:
        result = agent.print_stream_reply(student_reply, session=session)
    else:
        result = agent.reply(student_reply, session=session)
        print(result.content)

    print("student_reply:", student_reply)
    print("quality:", result.reply_quality.value)
    print("understands:", result.reply_analysis.understands)
    print("analysis_source:", result.reply_analysis.source)
    print("analysis_reason:", result.reply_analysis.reason)
    print("analysis_label:", ANALYSIS_SOURCE_LABELS.get(result.reply_analysis.source, "unknown"))
    print("mode:", result.strategy.mode.value)
    print("strategy_prompt:", result.strategy.prompt)
    print("turn_index:", result.turn_index)
    print("max_turns:", session.max_turns)
    print("done:", result.done)
    print("stop_reason:", result.stop_reason)
    print("-" * 50)
    return result

## Turn 1

学生完全没思路。

In [ ]:
r1 = run_turn("我不会。")

## Turn 2

学生开始意识到要先找公差。

In [ ]:
r2 = run_turn("是不是先求公差 d？")

## Turn 3

学生能用条件列式，但还没给出最终值。

In [ ]:
r3 = run_turn("因为 a_4=a_1+3d，所以 12=3+3d，先解 d。")

## Turn 4

学生完整做出题目。

In [ ]:
r4 = run_turn("先由 12=3+3d 解得 d=3，再用 a_10=a_1+9d=3+27=30。")

## Quick Summary

如果你想复测，可以重新创建一个新的 `session` 再跑 4 轮。